# Calibrating initial biomass and leaf lifespan in WOFOST 8.1 SNOMIN

This notebook recovers two crop parameters from a leaf-area series in a full
WOFOST 8.1 run with the SNOMIN soil-nitrogen model. The idea is the same as in
the WOFOST 7.2 potential-production notebook. The parameters are learnable
tensors, the loss compares simulated leaf area with a reference series, and
Adam follows the gradient of that loss.

`TDWI` is the total crop dry weight at emergence, in kg ha$^{-1}$. A larger
value starts the canopy higher. `SPAN` is the lifespan, in days, of leaves that
grew at 35 °C. A longer lifespan delays leaf death, so the canopy stays green
later in the season. Leaf area also controls assimilation, and assimilation
controls later growth, which is why both parameters have a gradient in the full
model. The day a leaf cohort dies is a hard threshold. While `SPAN` requires a
gradient, a straight-through estimator lets that threshold pass a derivative.
The forward simulation itself still uses the hard lifespan.

The reference series is not a field measurement. It is one run of this same
model at the crop-file values, so a successful search should return those
values. The search starts away from them: `SPAN` at 25 days instead of 35, and
`TDWI` at 200 kg ha$^{-1}$ instead of 593.

Run the notebook from the repository root. One season takes a few seconds on
CPU. A hundred optimization steps takes about fifteen minutes.

In [ ]:
import datetime as dt
import warnings
import inspect
import tempfile
import textwrap
from pathlib import Path

import torch
import yaml
from pcse.input import ExcelWeatherDataProvider
from pcse.input import WOFOST81SiteDataProvider_SNOMIN
from pcse.input import YAMLAgroManagementReader
from pcse.input import YAMLCropDataProvider
from pcse.models import Wofost81_NWLP_MLWB_SNOMIN
from pcse.soil.soil_profile import SoilProfile

from diffwofost.physical_models.config import ComputeConfig
from diffwofost.physical_models.config import Configuration
from diffwofost.physical_models.crop.wofost81 import Wofost81
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.parameter_providers import ParameterProvider
from diffwofost.physical_models.soil.soil_wrappers import SoilModuleWrapper_NWLP_MLWB_SNOMIN

ComputeConfig.set_dtype(torch.float64)
ComputeConfig.set_device("cpu")
warnings.filterwarnings("ignore", message="Converting a tensor with requires_grad=True to a scalar")

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parents[1]
WEATHER = ROOT / "docs/notebooks/data/weather_wageningen.xlsx"
SOWING = dt.date(2013, 10, 23)
SEASON_END = dt.date(2014, 7, 31)
CROP_NAME = "winterwheat"
VARIETY_NAME = "Arminda"
CROP_DIR = ROOT / "docs/notebooks/data/crop"

## Inputs

The crop is winter wheat variety `Arminda`, taken from the crop file used by
the NUE PCSE-Gym winter-wheat environment. In that file the initial biomass is
593 kg ha$^{-1}$ and `SPAN` is 35 days. An earlier trial with
`Winter_wheat_101` started from only 50 kg ha$^{-1}$ and never grew a canopy
larger than a leaf-area index of about 1.6, even with a large nitrogen supply.
`Arminda` reaches a peak near 5, which is a plausible winter-wheat canopy.

The soil profile is the worked example in the PCSE `SoilProfile` docstring.
Site parameters use the defaults of `WOFOST81SiteDataProvider_SNOMIN`, plus the
four values that provider requires: initial available water `WAV`, atmospheric
`CO2`, and the initial ammonium and nitrate in each soil layer.

Weather is the Wageningen record shipped with the PCSE notebook that runs
WOFOST 8.1 and SNOMIN on SoilGrids. It covers 1 January 2013 through
31 December 2015. Sowing is 23 October 2013, the autumn sowing date of the
winter-wheat example that uses this file, and the simulation stops at harvest
on 31 July 2014. That window includes winter, the spring sidedressings, and the
summer in which the leaves senesce, which is when `SPAN` becomes visible in the
leaf-area curve. The file stores minimum and maximum temperature. Daily mean
temperature is their average, and daytime temperature `DTEMP` is the average of
that mean and the maximum, as in the WOFOST 8.1 tests.

Nitrogen is applied as three synthetic dressings, 180 kg N ha$^{-1}$ in total.
Each dressing is 60 kg N ha$^{-1}$, half ammonium and half nitrate, worked into
the top 10 cm, with no organic matter. The first goes on at sowing. The other
two are sidedressings on 26 March and 29 April 2014, when a winter-wheat crop
is taking up nitrogen during stem elongation. Placing those two dressings a few
weeks after an October sowing would put them in winter, before the crop needs
them.

In [ ]:
def example_soil():
    lines = inspect.getdoc(SoilProfile).splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip() == "SoilLayerTypes:")
    end = next(i for i, line in enumerate(lines) if line.strip().startswith("GroundWater:"))
    parsed = yaml.safe_load(textwrap.dedent("\n".join(lines[start : end + 1])))
    profile = parsed["SoilProfileDescription"]
    if "FSOMI" not in profile["SubSoilType"]:
        profile["SubSoilType"]["FSOMI"] = 0.0
    rootable = sum(layer["Thickness"] for layer in profile["SoilLayers"])
    return {"RDMSOL": float(rootable), "SoilProfileDescription": profile}


def weather_rows(start, end):
    provider = ExcelWeatherDataProvider(WEATHER)
    names = (
        "DAY", "LAT", "LON", "ELEV", "IRRAD", "TMIN", "TMAX",
        "VAP", "WIND", "RAIN", "E0", "ES0", "ET0",
    )
    rows = []
    day = start
    while day <= end:
        container = provider(day)
        item = {name: getattr(container, name) for name in names}
        item["TEMP"] = 0.5 * (item["TMIN"] + item["TMAX"])
        item["DTEMP"] = 0.5 * (item["TEMP"] + item["TMAX"])
        rows.append(item)
        day += dt.timedelta(days=1)
    return rows


def agro(start):
    harvest = start + dt.timedelta(days=300)
    dressings = (
        (start, 60),
        (dt.date(2014, 3, 26), 60),
        (dt.date(2014, 4, 29), 60),
    )
    events = "\n".join(
        f"""        - {day.isoformat()}:
            amount: {amount}
            application_depth: 10
            cnratio: 0
            initial_age: 0
            f_NH4N: 0.5
            f_NO3N: 0.5
            f_orgmat: 0"""
        for day, amount in dressings
    )
    with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as handle:
        path = Path(handle.name)
        handle.write(
            f"""
AgroManagement:
- {start.isoformat()}:
    CropCalendar:
        crop_name: {CROP_NAME}
        variety_name: {VARIETY_NAME}
        crop_start_date: {start.isoformat()}
        crop_start_type: sowing
        crop_end_date: {harvest.isoformat()}
        crop_end_type: harvest
        max_duration: 300
    TimedEvents:
      - event_signal: apply_n_snomin
        name: Synthetic nitrogen
        comment: amount is kg N/ha because the amendment is entirely mineral
        events_table:
{events}
    StateEvents: null
"""
        )
    try:
        return YAMLAgroManagementReader(str(path))
    finally:
        path.unlink(missing_ok=True)


rows = weather_rows(SOWING, SEASON_END)
crop = YAMLCropDataProvider(Wofost81_NWLP_MLWB_SNOMIN, fpath=CROP_DIR, force_reload=True)
soil = example_soil()
n_layers = len(soil["SoilProfileDescription"]["SoilLayers"])
site = WOFOST81SiteDataProvider_SNOMIN(
    WAV=20.0, CO2=360.0, NH4I=[10.0] * n_layers, NO3I=[1.0] * n_layers
)
campaign = agro(rows[0]["DAY"])
params = ParameterProvider(cropdata=crop, soildata=soil, sitedata=site)
params.set_active_crop(CROP_NAME, VARIETY_NAME, "sowing", "harvest")
true_span = float(params["SPAN"])
true_tdwi = float(params["TDWI"])
print(
    f"Crop-file SPAN = {true_span:.2f} days, TDWI = {true_tdwi:.1f} kg/ha. "
    f"Simulation = {rows[0]['DAY']} to {rows[-1]['DAY']}."
)

In [ ]:
config = Configuration(
    CROP=Wofost81,
    SOIL=SoilModuleWrapper_NWLP_MLWB_SNOMIN,
    OUTPUT_VARS=["LAI", "TWLV", "DVS"],
)


def run(span, tdwi):
    """One SNOMIN season. Either argument may be a tensor that requires a gradient."""
    params.set_override("SPAN", span, check=False)
    params.set_override("TDWI", tdwi, check=False)
    model = Engine(config)
    model.setup(params, iter(rows), campaign)
    model.run_till(rows[-1]["DAY"])
    output = model.get_output()
    return torch.stack([day["LAI"].reshape(()) for day in output])


with torch.no_grad():
    observed = run(
        torch.tensor(true_span, dtype=torch.float64),
        torch.tensor(true_tdwi, dtype=torch.float64),
    ).detach()
print(f"Observed LAI on the last day: {observed[-1].item():.3f}")

## Calibration

`SPAN` and `TDWI` are fitted together against leaf area. Each parameter is
kept inside a fixed range by storing it as a logit and mapping that logit
through a sigmoid, which is the `BoundedParameter` used in the other
optimization notebooks. `SPAN` is limited to 10–60 days and starts at 25.
`TDWI` is limited to 50–1200 kg ha$^{-1}$, which contains the Arminda value of
593, and starts at 200. The loss is the mean absolute error of daily `LAI`.

The two parameters can stand in for each other. A longer leaf life raises late
leaf area in much the same way as a larger biomass at emergence raises early
leaf area, and both change the growth that follows. Early in the search,
`SPAN` can therefore walk past 35 while `TDWI` is still well below 593. With
about a hundred Adam steps the loss comes back down and both parameters arrive
close to the crop file. The printed gradient is the derivative of the loss with
respect to the unbounded logit, not with respect to the parameter in days or
kg ha$^{-1}$.

In [ ]:
class BoundedParameter(torch.nn.Module):
    def __init__(self, low, high, init_value):
        super().__init__()
        self.low = low
        self.high = high
        init_norm = (init_value - low) / (high - low)
        self.raw = torch.nn.Parameter(
            torch.logit(torch.tensor(init_norm, dtype=torch.float64), eps=1e-6)
        )

    def forward(self):
        return self.low + (self.high - self.low) * torch.sigmoid(self.raw)


SPAN_MIN, SPAN_MAX, SPAN_INIT = 10.0, 60.0, 25.0
TDWI_MIN, TDWI_MAX, TDWI_INIT = 50.0, 1200.0, 200.0
span = BoundedParameter(SPAN_MIN, SPAN_MAX, SPAN_INIT)
tdwi = BoundedParameter(TDWI_MIN, TDWI_MAX, TDWI_INIT)
optimizer = torch.optim.Adam(list(span.parameters()) + list(tdwi.parameters()), lr=0.05)

loss_history = []
span_history = []
tdwi_history = []

for step in range(100):
    optimizer.zero_grad()
    simulated = run(span(), tdwi())
    loss = torch.mean(torch.abs(simulated - observed))
    loss.backward()
    span_gradient = span.raw.grad
    tdwi_gradient = tdwi.raw.grad
    optimizer.step()
    loss_history.append(loss.item())
    span_history.append(span().item())
    tdwi_history.append(tdwi().item())
    print(
        f"Step {step}, loss {loss.item():.4f}, "
        f"SPAN {span().item():.2f}, TDWI {tdwi().item():.1f}, "
        f"grads {span_gradient.item():.4e} {tdwi_gradient.item():.4e}"
    )

print(
    f"Crop-file SPAN {true_span:.2f}, TDWI {true_tdwi:.1f}. "
    f"Calibrated SPAN {span().item():.2f}, TDWI {tdwi().item():.1f}."
)

## Results

The first figure is the calibration itself. The upper panel is the leaf-area
error at each Adam step. The lower panel shows the two parameters, with dotted
lines at the crop-file values. A useful run drives the error down and brings
both curves onto those lines. Because the parameters trade off, the leaf-area
error can rise again for a while after `SPAN` has already passed 35, and then
fall once `TDWI` catches up.

The second figure is the season. It compares the reference leaf-area curve,
made with the crop-file parameters, with the curve from the parameters at the
end of the search. Days are counted from sowing on 23 October 2013. If the
search has recovered the crop file, the two curves lie on top of each other.

In [ ]:
import matplotlib.pyplot as plt

steps = range(len(loss_history))
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

axes[0].plot(steps, loss_history, linewidth=2)
axes[0].set_ylabel("LAI mean absolute error")
axes[0].set_title("Loss during calibration")
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, span_history, label="SPAN (days)", linewidth=2)
axes[1].axhline(true_span, color="C0", linestyle=":", label=f"Crop-file SPAN {true_span:.0f}")
tdwi_axis = axes[1].twinx()
tdwi_axis.plot(steps, tdwi_history, color="C1", linestyle="--", label="TDWI (kg/ha)", linewidth=2)
tdwi_axis.axhline(true_tdwi, color="C1", linestyle=":")
axes[1].set_xlabel("Optimization step")
axes[1].set_ylabel("SPAN (days)")
tdwi_axis.set_ylabel("TDWI (kg ha$^{-1}$)")
axes[1].set_title("Calibrated parameters")
handles, labels = axes[1].get_legend_handles_labels()
tdwi_handles, tdwi_labels = tdwi_axis.get_legend_handles_labels()
axes[1].legend(handles + tdwi_handles, labels + tdwi_labels, loc="center right")
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"Calibrated: SPAN = {span().item():.2f}, TDWI = {tdwi().item():.1f}")
print(f"Crop file:  SPAN = {true_span:.2f}, TDWI = {true_tdwi:.1f}")

In [ ]:
with torch.no_grad():
    optimized_lai = run(span().detach(), tdwi().detach()).numpy()

reference_lai = observed.numpy()
days = range(len(reference_lai))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(days, reference_lai, label="Reference LAI", linewidth=2)
ax.plot(days, optimized_lai, label="Optimized LAI", linewidth=2, linestyle="--")
ax.set_xlabel("Time step (days after sowing)")
ax.set_ylabel("LAI (m² m⁻²)")
ax.set_title("Reference vs. optimized LAI")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()